# 09 - Cross Validation

In this notebook, we explore why a simple Train/Test split is dangerous, and how Cross-Validation provides a robust estimate of model performance.

## Concept Overview
If you have a small dataset, a random split might accidentally put all the 'easy' examples in the Train set and all the 'hard' examples in the Test set. Your model will look terrible. K-Fold Cross Validation solves this by rotating the Test set.

## Visualizing the Problem
Let's see how much the accuracy jumps around just by changing the `random_state` of the Train/Test split.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load dataset (569 samples)
data = load_breast_cancer()
X, y = data.data, data.target

scores = []
for random_seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    scores.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(range(20), scores, marker='o')
plt.axhline(np.mean(scores), color='r', linestyle='--', label=f'Mean: {np.mean(scores):.2f}')
plt.title('Accuracy Variance due to Random Splitting')
plt.xlabel('Random Seed')
plt.ylabel('Accuracy on Test Set')
plt.legend()
plt.show()

Notice how the accuracy bounces between 88% and 96%! Which one is the 'true' accuracy? This is why we need Cross Validation.

## Scikit-Learn Implementation: K-Fold CV
We will use `cross_val_score` to automatically handle the splitting, training, and scoring.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

# 1. Standard K-Fold (Random splits, ignores class imbalance)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

# 2. Stratified K-Fold (Preserves the % of cancer vs healthy in every fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print(f'Standard K-Fold Average:   {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}')
print(f'Stratified K-Fold Average: {skf_scores.mean():.4f} +/- {skf_scores.std():.4f}')

For classification tasks, **always use StratifiedKFold**. Scikit-learn actually uses StratifiedKFold by default if you just pass `cv=5` to `cross_val_score` for a classification model.

## Advanced: Time Series Split
Never use K-Fold on time series data, or you will train on the future to predict the past.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

X_time = np.arange(10)

tscv = TimeSeriesSplit(n_splits=3)
print("Time Series Splits:")
for i, (train_index, test_index) in enumerate(tscv.split(X_time)):
    print(f"Fold {i}: Train={train_index}, Test={test_index}")

## Industry Notes
When building models for Kaggle or Production, Data Scientists often build their validation strategy first. If your CV score doesn't correlate with your real-world test score, your validation strategy is broken (likely due to data leakage).

## Exercises
1. Try running `cross_validate` instead of `cross_val_score`. It returns a dictionary. Extract the average training time (`fit_time`) for the folds.
2. Implement `LeaveOneOut` CV on a tiny subset of the data (e.g., 50 rows). How many folds does it create?